# Fase 2 (lanjutan) — Audit Kualitas Data

Notebook ini menjalankan pengecekan duplikasi, referential integrity, dan distribusi nilai langsung di `olist_db`, sebagai bagian akhir Fase 2 sebelum lanjut ke Fase 3/4.

**Prasyarat:** `01_load_to_postgres.ipynb` sudah selesai jalan dengan semua status `OK`.

In [1]:
import os
import getpass
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text, URL
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("../.env"))

DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "olist_db")

if not DB_PASSWORD:
    DB_PASSWORD = getpass.getpass("Password PostgreSQL: ")

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER, password=DB_PASSWORD,
    host=DB_HOST, port=int(DB_PORT), database=DB_NAME,
)
engine = create_engine(url)

def q(sql):
    return pd.read_sql(text(sql), engine)

## 1. Cek Duplikasi

> **Temuan (dari analisis CSV):** `review_id` di `order_reviews` punya **814 baris duplikat (0.82%)** — dataset asli Olist memang punya kasus ini. Kolom kunci komposit di `order_items` (`order_id`+`order_item_id`) dan `order_payments` (`order_id`+`payment_sequential`) **bersih, tidak ada duplikasi**.

In [2]:
q("""
SELECT review_id, COUNT(*) AS jumlah
FROM order_reviews
GROUP BY review_id
HAVING COUNT(*) > 1
ORDER BY jumlah DESC;
""")

,review_id,jumlah
0,1fb4ddc969e6bea80e38deec00393a6f,3
1,38821b5c496b678cf91acc34892805ad,3
2,308316408775d1600dad81bd3184556d,3
3,44e9f871226d8a130de3fc39dfbdf0c5,3
4,e44840754f12fad2b8646712121b349a,3
...,...,...
784,6c3c56231c45f130228a386e4d9b124a,2
785,9a07cb8136a1792e7984d827e2336d60,2
786,b50b479cf97c8998967015fac6825c58,2
787,e7452eb647855255b2045780c7e90097,2


In [3]:
# Sanity check composite key order_items & order_payments (harus kosong)
print("Duplikat order_items:")
print(q("""
SELECT order_id, order_item_id, COUNT(*) FROM order_items
GROUP BY order_id, order_item_id HAVING COUNT(*) > 1;
"""))

print("\nDuplikat order_payments:")
print(q("""
SELECT order_id, payment_sequential, COUNT(*) FROM order_payments
GROUP BY order_id, payment_sequential HAVING COUNT(*) > 1;
"""))

Duplikat order_items:
Empty DataFrame
Columns: [order_id, order_item_id, count]
Index: []

Duplikat order_payments:
Empty DataFrame
Columns: [order_id, payment_sequential, count]
Index: []


## 2. Referential Integrity (Orphan Records)

> **Temuan:** Semua relasi antar tabel **bersih (0 orphan)** — `orders`→`customers`, `order_items`→`orders`/`products`/`sellers`, `order_payments`→`orders`, `order_reviews`→`orders`. **Kecuali satu:** 13 baris di `products.product_category_name` (yang bukan NULL) tidak ditemukan padanannya di `product_category_translation` — kemungkinan typo/kategori baru yang belum di-translate.

In [4]:
checks = {
    "orders.customer_id -> customers": """
        SELECT COUNT(*) AS orphan FROM orders o
        LEFT JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL""",
    "order_items.order_id -> orders": """
        SELECT COUNT(*) AS orphan FROM order_items oi
        LEFT JOIN orders o ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL""",
    "order_items.product_id -> products": """
        SELECT COUNT(*) AS orphan FROM order_items oi
        LEFT JOIN products p ON oi.product_id = p.product_id
        WHERE p.product_id IS NULL""",
    "order_items.seller_id -> sellers": """
        SELECT COUNT(*) AS orphan FROM order_items oi
        LEFT JOIN sellers s ON oi.seller_id = s.seller_id
        WHERE s.seller_id IS NULL""",
    "order_payments.order_id -> orders": """
        SELECT COUNT(*) AS orphan FROM order_payments op
        LEFT JOIN orders o ON op.order_id = o.order_id
        WHERE o.order_id IS NULL""",
    "order_reviews.order_id -> orders": """
        SELECT COUNT(*) AS orphan FROM order_reviews r
        LEFT JOIN orders o ON r.order_id = o.order_id
        WHERE o.order_id IS NULL""",
    "products.product_category_name -> translation": """
        SELECT COUNT(*) AS orphan FROM products p
        LEFT JOIN product_category_translation t
            ON p.product_category_name = t.product_category_name
        WHERE p.product_category_name IS NOT NULL AND t.product_category_name IS NULL""",
}

for label, sql in checks.items():
    result = q(sql).iloc[0, 0]
    flag = "OK" if result == 0 else f"ADA {result} ORPHAN"
    print(f"{label:45s} -> {flag}")

orders.customer_id -> customers               -> OK
order_items.order_id -> orders                -> OK
order_items.product_id -> products            -> OK
order_items.seller_id -> sellers              -> OK
order_payments.order_id -> orders             -> OK
order_reviews.order_id -> orders              -> OK
products.product_category_name -> translation -> ADA 13 ORPHAN


## 3. Distribusi Kategori / Status

> **Temuan:**
> - `order_status`: 8 nilai, **97% `delivered`** — untuk analisis waktu pengiriman & revenue, sebaiknya filter hanya status `delivered` supaya tidak bias oleh order yang batal/belum selesai
> - `payment_type`: 5 nilai, didominasi `credit_card` (76.795) dan `boleto` (19.784) — ada 3 baris `not_defined` (anomali kecil, bisa di-exclude)
> - `review_score`: rentang 1–5, distribusi condong ke skor tinggi (57.328 dapat skor 5, hanya 3.151 dapat skor 2) — perlu diingat saat interpretasi rata-rata skor

In [5]:
print("order_status:")
print(q("SELECT order_status, COUNT(*) FROM orders GROUP BY order_status ORDER BY COUNT(*) DESC"))

print("\npayment_type:")
print(q("SELECT payment_type, COUNT(*) FROM order_payments GROUP BY payment_type ORDER BY COUNT(*) DESC"))

print("\nreview_score:")
print(q("SELECT review_score, COUNT(*) FROM order_reviews GROUP BY review_score ORDER BY review_score"))

order_status:
  order_status  count
0    delivered  96478
1      shipped   1107
2     canceled    625
3  unavailable    609
4     invoiced    314
5   processing    301
6      created      5
7     approved      2

payment_type:
  payment_type  count
0  credit_card  76795
1       boleto  19784
2      voucher   5775
3   debit_card   1529
4  not_defined      3

review_score:
   review_score  count
0             1  11424
1             2   3151
2             3   8179
3             4  19142
4             5  57328


## 4. Order Tanpa order_items

> **Temuan:** **775 order** tidak punya baris `order_items` sama sekali — mayoritas berstatus `unavailable` (603) dan `canceled` (164). Ini masuk akal (order dibatalkan sebelum item diproses), tapi **penting untuk Fase 4**: query revenue/RFM yang JOIN ke `order_items` otomatis akan mengecualikan 775 order ini — itu perilaku yang benar, bukan bug, tapi perlu disebutkan di dokumentasi metodologi.

In [6]:
q("""
SELECT o.order_status, COUNT(*) AS jumlah
FROM orders o
LEFT JOIN order_items oi ON o.order_id = oi.order_id
WHERE oi.order_id IS NULL
GROUP BY o.order_status
ORDER BY jumlah DESC;
""")

,order_status,jumlah
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


## Ringkasan Temuan & Implikasi untuk Fase 4

| Temuan | Implikasi |
|---|---|
| 814 `review_id` duplikat | Dedup dulu (`DISTINCT ON` atau `ROW_NUMBER()`) sebelum JOIN review ke analisis lain |
| 13 kategori produk tidak match translation | Pakai `LEFT JOIN` (bukan `INNER JOIN`) ke translation table, kategori itu akan tampil sebagai NULL/uncategorized |
| 775 order tanpa `order_items` | Wajar untuk order batal/unavailable — otomatis ter-exclude saat JOIN ke `order_items`, catat di metodologi |
| Referential integrity lain | Semua bersih — aman untuk JOIN multi-tabel tanpa filter tambahan |
| `order_status` 97% delivered | Filter `WHERE order_status = 'delivered'` untuk analisis waktu kirim & revenue realized |

**Fase 2 selesai.** Lanjut ke Fase 3 (kalau ada transformasi/cleaning tambahan yang diperlukan) atau langsung Fase 4 (SQL query development: RFM, revenue trend, churn, kategori & wilayah).